In [0]:
%pip install aiohttp beautifulsoup4 lxml pandas
dbutils.library.restartPython()

In [0]:
import pandas as pd
# Importamos la biblioteca necesaria para hacer peticiones HTTP asíncronas
import asyncio
import aiohttp
import pandas as pd
import re
from bs4 import BeautifulSoup

In [0]:
BASE_URL = "https://books.toscrape.com/"

In [0]:

# Definimos una función asíncrona indicando la URL a la que queremos acceder
async def fetch_html(url):
    
    # Abrimos una sesión HTTP. 'async with' garantiza que los recursos 
    # se liberen y la sesión se cierre automáticamente al terminar.
    async with aiohttp.ClientSession() as session:
        
        # Realizamos la petición HTTP tipo GET a la URL indicada
        async with session.get(url) as response:
            
            # Verificamos que la respuesta sea exitosa (código 200). 
            # Si hay un error (ej. 404 No encontrado), esto lanza una excepción.
            response.raise_for_status()
            
            # Esperamos asíncronamente a que se descargue el cuerpo de la página 
            # y lo devolvemos en formato de texto (string).
            return await response.text()

In [0]:
# Hacer petición de forma asíncrona
html_content = await fetch_html("https://books.toscrape.com/")
print(html_content)

In [0]:
# Importamos la clase BeautifulSoup desde la biblioteca bs4
from bs4 import BeautifulSoup

# Creamos el objeto 'soup. Le pasamos dos cosas:
# 1. 'html_content': la variable que contiene todo el texto HTML (lo que descargamos antes).
# 2. '"lxml"': es el motor que lee y estructura el HTML. Se usa porque es muy rápido y eficiente.
soup = BeautifulSoup(html_content, "lxml")

# Navegamos por la estructura del HTML hasta la etiqueta <title>.
# Al agregar '.text', le decimos que ignore las etiquetas HTML (<title>...</title>) 
# y nos devuelva únicamente el texto limpio que está adentro. Finalmente, lo imprimimos.
print(soup.title.text)

In [0]:
# Creamos una lista vacía donde guardaremos la información de cada libro.
books = []

# 'soup.select' usa selectores CSS.
# Aquí busca TODAS las etiquetas <article> que tengan la clase "product_pod".
# El bucle 'for' va a recorrer cada uno de esos artículos (cada artículo es un libro).
for article in soup.select("article.product_pod"):
    
    # Dentro de ese artículo específico, busca la etiqueta <h3>, luego la etiqueta <a> (enlace),
    # y extrae el valor del atributo "title" (que contiene el nombre completo del libro).
    title = article.h3.a["title"]
    
    # 'select_one' busca solo el PRIMER elemento que coincida con el selector CSS.
    # Aquí busca algo con la clase "price_color", extrae su texto y usa '.strip()' 
    # para limpiar cualquier espacio en blanco o salto de línea al principio y al final.
    price_text = article.select_one(".price_color").text.strip()
    
    # Hace exactamente lo mismo que el precio, pero buscando la clase "availability".
    availability = article.select_one(".instock").text.strip()
    
    # Crea un diccionario con los datos limpios de este libro en particular 
    # y lo añade (append) a nuestra lista 'books'.
    books.append({
        "product_name": title,
        "price_text": price_text,
        "availability": availability,
        "source": "books_to_scrape" # Añade un dato fijo para saber de dónde salió la info
    })

# Imprime en consola cuántos diccionarios (libros) se guardaron en la lista en total.
print(f"Libros encontrados: {len(books)}")

# Muestra los elementos de la lista
books

In [0]:
df = pd.DataFrame(books)
df

In [0]:
df["price"] = df["price_text"].str.extract(r'([\d.]+)').astype(float)

In [0]:
df["currency"] = "GBP"

df


In [0]:
df["in_stock"] = df["availability"].str.contains("In stock", case=False)
df.head()

In [0]:
df = df[["product_name", "price", "currency", "in_stock", "source"]]
df.head()

In [0]:

# --- 1. FASE DE DESCARGA (ASÍNCRONA) ---
async def fetch_page(session, url, semaphore):
    """Descarga el HTML de una página de forma segura, respetando el límite de peticiones."""
    
    # El 'semaphore' actúa como un embudo. Si ya hay 5 peticiones corriendo, 
    # pone esta en espera hasta que una termine. Esto evita bloqueos del servidor.
    async with semaphore:
        # Usamos try/except para que, si una página de las 50 falla, 
        # el programa no se detenga por completo.
        try:
            # Hacemos la petición GET a la URL usando la sesión compartida
            async with session.get(url) as response:
                # Verifica que la respuesta sea un éxito (código 200). 
                # Si es un error 404 o 500, lanza una excepción para ir al bloque 'except'.
                response.raise_for_status()
                # Extrae y devuelve el código HTML en formato texto.
                return await response.text()
        except Exception as e:
            # Si algo falla, imprime el error y devuelve None en lugar de romper el código.
            print(f"Error al descargar {url}: {e}")
            return None

# --- 2. FASE DE EXTRACCIÓN ---
def parse_books(html):
    """Recibe un HTML crudo, usa BeautifulSoup y devuelve una lista de diccionarios con los libros."""
    
    # Si la fase anterior falló y devolvió None, aquí devolvemos una lista vacía.
    if not html:
        return []
    
    # Convertimos el texto HTML en un objeto navegable usando el motor rápido 'lxml'
    soup = BeautifulSoup(html, "lxml")
    books_on_page = []
    
    # Buscamos todos los bloques HTML que representen un libro en la página
    for article in soup.select("article.product_pod"):
        # Extraemos el título del atributo 'title' dentro del enlace <a>
        title = article.h3.a["title"]
        # Buscamos el precio, extraemos su texto y limpiamos los espacios en blanco
        price_text = article.select_one(".price_color").text.strip()
        # Buscamos el texto de disponibilidad y también lo limpiamos
        availability = article.select_one(".availability").text.strip()
        
        # Guardamos los datos de este libro específico en un diccionario 
        # y lo agregamos a la lista de libros de esta página.
        books_on_page.append({
            "product_name": title,
            "price_text": price_text,
            "availability": availability,
            "source": "books_to_scrape"
        })
        
    return books_on_page

# --- 3. ORQUESTADOR PRINCIPAL ---
async def main():
    print("Iniciando la extracción de 50 páginas...")
    
    # 3.1 PREPARACIÓN
    # Creamos un patrón base para la URL donde '{}' será reemplazado por el número de página.
    base_url = "http://books.toscrape.com/catalogue/page-{}.html"
    # Generamos una lista con las 50 URLs (del 1 al 50) usando comprensión de listas.
    urls = [base_url.format(i) for i in range(1, 51)]
    
    # Establecemos el límite máximo de peticiones simultáneas a 5.
    semaphore = asyncio.Semaphore(5)
    
    # 3.2 EJECUCIÓN ASÍNCRONA
    # Abrimos una única sesión HTTP que se reutilizará para todas las descargas (es más rápido).
    async with aiohttp.ClientSession() as session:
        # Preparamos las 50 "tareas" de descarga, pero aún no las ejecutamos.
        tasks = [fetch_page(session, url, semaphore) for url in urls]
        
        # 'asyncio.gather' dispara todas las tareas al mismo tiempo (respetando el semáforo)
        # y espera a que todas terminen para guardar los HTML en una lista.
        html_pages = await asyncio.gather(*tasks)
    
    print("Descarga completada. Procesando los datos...")
    
    # 3.3 EXTRACCIÓN DE DATOS
    all_books = []
    # Recorremos los 50 HTML descargados
    for html in html_pages:
        # Extraemos los libros de cada HTML y los "fusionamos" en la lista maestra (extend).
        all_books.extend(parse_books(html))
        
    print(f"Total de libros extraídos: {len(all_books)}")

    # --- 4. FASE DE LIMPIEZA CON PANDAS ---
    # Convertimos la lista maestra de diccionarios en una tabla estructurada (DataFrame)
    df = pd.DataFrame(all_books)
    
    # Limpiamos el precio: usamos una expresión regular para sacar solo números y puntos,
    # y lo convertimos a un valor numérico real (float) para poder hacer cálculos después.
    df["price"] = df["price_text"].str.extract(r'([\d.]+)').astype(float)
    # Agregamos la moneda para no perder ese contexto.
    df["currency"] = "GBP"
    # Verificamos si el texto original contenía "In stock" y devolvemos True o False.
    df["in_stock"] = df["availability"].str.contains("In stock", case=False)
    
    # Filtramos la tabla para quedarnos únicamente con las columnas limpias y útiles.
    df = df[["product_name", "price", "currency", "in_stock", "source"]]
    
    print("¡Éxito! Los datos han sido estructurados.")
    
    # Retornamos el DataFrame final para que pueda ser capturado fuera de esta función.
    return df

# --- 5. EJECUCIÓN EN DATABRICKS ---
# Como Databricks ya tiene un bucle de eventos corriendo, usamos 'await' en lugar de 'asyncio.run()'.
# El resultado (nuestra tabla) queda guardado en la variable 'df_books'.
df_books = await main()

# Ahora puedes visualizar la tabla de forma interactiva en la siguiente celda usando:
df_books.head()

In [0]:
df_books.shape

In [0]:
books_spark_df = spark.createDataFrame(df_books)

books_spark_df.write.mode("overwrite").format("delta").saveAsTable("bronze.web_books")

In [0]:
%sql
-- select * from bronze.web_books
select count(1) from bronze.web_books

In [0]:
%sql
select * from bronze.web_books